In [ ]:
import pandas as pd 
frames_df = pd.read_json("all_facial_metrics.json")  
frames_df.describe()

In [ ]:
frames_df['img_path'].unique()

In [ ]:
#extract the timestamp using regex 
import re 
frames_df['timestamp'] = frames_df['img_path'].apply(lambda x: int(re.search(r'frame_(\d+)', str(x)).group(1)))
# since our fps was 1/s , leave it so it is
import pandas as pd 
audio_df = pd.read_csv("audio_features_df.csv")

In [ ]:
frames_df.rename(columns={'dj': 'video_name'}, inplace=True)


In [ ]:
frames_df['video_name'].value_counts()

In [ ]:
import difflib

# Get the unique names as lists
audio_videos = audio_df['video_name'].dropna().unique().tolist()
frame_videos = frames_df['video_name'].dropna().unique().tolist()

print("--- String Matching Suggestions ---\n")
for audio_vid in audio_videos:
    # Find the single closest match in the frames dataframe (cutoff controls how fuzzy it is)
    closest_match = difflib.get_close_matches(audio_vid, frame_videos, n=1, cutoff=0.3)
    
    if closest_match:
        if audio_vid == closest_match[0]:
            pass # They matched exactly, no action needed!
        else:
            print(f"⚠️ Partial match found! You should probably rename this:")
            print(f"   In Audio DF : '{audio_vid}'")
            print(f"   In Frames DF: '{closest_match[0]}'\n")
    else:
        print(f" No similar name found in Frames DF for: '{audio_vid}'\n")


In [ ]:
audio_df.rename(columns={'timestamp_start': 'timestamp'}, inplace=True)


In [ ]:
audio_df

In [ ]:
# now the interesting part regarding RL: our agent will learn based on those combined metrics
# for this exercise we can simply check for each timelapse of a song what was the percentage of happy sad neutral and normalise it, thats our reward 
# merge, cut based on track timestamp, get the metrics 

audio_df = audio_df.sort_values('timestamp')
frames_df = frames_df.sort_values('timestamp')
# double join on the dj 
frames_df_reward = pd.merge_asof(
    audio_df, 
    frames_df, 
    on='timestamp',       # closest
    by='video_name',              # exact
    direction='nearest', 
    tolerance=4
)

In [ ]:
frames_df_reward['label'].notna().value_counts